# Paper figure layout: synthetic
Snapshot of the approved drawing source. Embedded positions/edge lists render the saved figure; use benchmark renderers for newly trained graphs.


In [ ]:
from pathlib import Path
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/paper_protocol.py").is_file())
FIGURE_DIR = PROJECT_ROOT / "outputs" / "paper_figures" / "synthetic"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


real water

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path



pos = {
    0: (0.000000, 0.757412),
    1: (0.067536, 0.797539),
    2: (0.092528, 0.641360),
    3: (0.121592, 0.716384),
    4: (0.126754, 0.688650),
    5: (0.139235, 0.722846),
    6: (0.166247, 0.241877),
    7: (0.183939, 0.751907),
    8: (0.193977, 0.390470),
    9: (0.209903, 0.838518),
    10: (0.224710, 0.379113),
    11: (0.231066, 0.312292),
    12: (0.240956, 0.337722),
    13: (0.243716, 0.252634),
    14: (0.260438, 0.918997),
    15: (0.262816, 0.812384),
    16: (0.280593, 0.354880),
    17: (0.297833, 0.301472),
    18: (0.303537, 0.363794),
    19: (0.303950, 0.023104),
    20: (0.308506, 0.148358),
    21: (0.309541, 0.245172),
    22: (0.333527, 0.118148),
    23: (0.375724, 0.147498),
    24: (0.376258, 0.161106),
    25: (0.382201, 0.973422),
    26: (0.400571, 0.059632),
    27: (0.411097, 1.000000),
    28: (0.447922, 0.119139),
    29: (0.466120, 0.000000),
    30: (0.472562, 0.924592),
    31: (0.485747, 0.047209),
    32: (0.492203, 0.892988),
    33: (0.501980, 0.930939),
    34: (0.505856, 0.950283),
    35: (0.508124, 0.968420),
    36: (0.536860, 0.105678),
    37: (0.541557, 0.975983),
    38: (0.552445, 0.936794),
    39: (0.557386, 0.869792),
    40: (0.702212, 0.695793),
    41: (0.725256, 0.556286),
    42: (0.747499, 0.579026),
    43: (0.778713, 0.575698),
    44: (0.807738, 0.522943),
    45: (0.827076, 0.695128),
    46: (0.850785, 0.573348),
    47: (0.884430, 0.530074),
    48: (0.914390, 0.551015),
    49: (0.993087, 0.686943),
}

# ============================================================
# ARCHI DEL GRAFO GROUND TRUTH
# Gli ID eXX sono definiti ESATTAMENTE dall'ordine seguente.
# ============================================================

# Ground-truth edges. Edge IDs eXX are defined by this exact order.
gt_edges = [
    (0, 1), (1, 3), (2, 3), (2, 4), (3, 4), (3, 5),
    (3, 7), (5, 7), (6, 13), (6, 20), (7, 9), (7, 11),
    (7, 33), (8, 10), (9, 14), (9, 15), (9, 25), (10, 12),
    (11, 12), (11, 13), (11, 16), (11, 17), (11, 26), (12, 16),
    (16, 17), (16, 18), (17, 21), (19, 26), (20, 22), (20, 24),
    (21, 24), (21, 28), (22, 23), (23, 24), (23, 28), (25, 27),
    (26, 28), (27, 30), (28, 31), (28, 36), (29, 31), (30, 32),
    (30, 33), (30, 34), (31, 36), (32, 39), (33, 34), (33, 46),
    (34, 35), (34, 37), (35, 37), (37, 38), (40, 45), (41, 42),
    (41, 44), (42, 43), (43, 44), (44, 46), (44, 47), (47, 48),(48, 49), (45, 46),
   
]

# ============================================================
# COSTRUZIONE DEL GRAFO GROUND TRUTH
# ============================================================

G_gt = nx.Graph()

# Aggiunge tutti i nodi, anche quelli eventualmente isolati.
G_gt.add_nodes_from(pos.keys())
G_gt.add_edges_from(gt_edges)

print(f"Nodi ground truth: {G_gt.number_of_nodes()}")
print(f"Archi ground truth: {G_gt.number_of_edges()}")
print(f"Ground truth connessa: {nx.is_connected(G_gt)}")


In [ ]:
def canonical_edge(edge):
    """Restituisce una rappresentazione stabile di un arco non orientato."""
    if len(edge) != 2:
        raise ValueError(f"Un arco deve contenere esattamente due nodi: {edge}")

    u, v = edge

    # Mantiene l'ordine numerico per nodi interi e l'ordine naturale
    # per identificativi confrontabili. Il fallback gestisce tipi misti.
    try:
        return tuple(sorted((u, v)))
    except TypeError:
        return tuple(
            sorted((u, v), key=lambda node: (type(node).__name__, repr(node)))
        )


# ============================================================
# CATALOGO DEGLI ARCHI REALI
# L'ordine resta quello dichiarato in gt_edges.
# Quindi e00, e01, ... coincidono con i commenti della cella precedente.
# ============================================================

edge_catalog = {
    f"e{i:02d}": canonical_edge(edge)
    for i, edge in enumerate(gt_edges)
}

# Controllo cautelativo: nella ground truth non devono esserci duplicati.
if len(set(edge_catalog.values())) != len(edge_catalog):
    raise ValueError("La lista gt_edges contiene archi duplicati.")

# Formato richiesto da NetworkX: arco -> etichetta.
edge_labels = {
    edge: edge_id
    for edge_id, edge in edge_catalog.items()
}


def plot_edge_catalog(
    graph,
    positions,
    labels,
    figsize=(18, 10),
    node_size=28,
    edge_width=0.8,
    edge_label_font_size=6,
    show_node_labels=True,
    save_path=None,
):
    """
    Disegna il grafo ground truth mostrando:
    - l'identificativo eXX vicino a ogni arco reale;
    - opzionalmente, l'identificativo di ciascun nodo.

    Le etichette dei nodi permettono di scegliere manualmente anche
    gli estremi degli eventuali falsi positivi.
    """

    fig, ax = plt.subplots(figsize=figsize)

    nx.draw_networkx_edges(
        graph,
        positions,
        ax=ax,
        width=edge_width,
        edge_color="0.30",
    )

    nx.draw_networkx_nodes(
        graph,
        positions,
        ax=ax,
        node_size=node_size,
    )

    if show_node_labels:
        nx.draw_networkx_labels(
            graph,
            positions,
            ax=ax,
            font_size=6,
            verticalalignment="bottom",
        )

    nx.draw_networkx_edge_labels(
        graph,
        positions,
        edge_labels=labels,
        ax=ax,
        font_size=edge_label_font_size,
        rotate=False,
        label_pos=0.5,
        bbox={
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.85,
            "pad": 0.8,
        },
    )

    ax.set_aspect("equal")
    ax.axis("off")
    plt.tight_layout()

    # DECOMMENTARE SOLO QUANDO SI VUOLE SALVARE LA FIGURA
    # if save_path is not None:
    #     save_path = Path(save_path)
    #     save_path.parent.mkdir(parents=True, exist_ok=True)
    #     fig.savefig(
    #         save_path,
    #         dpi=300,
    #         bbox_inches="tight",
    #         facecolor="white",
    #     )

    plt.show()


plot_edge_catalog(
    graph=G_gt,
    positions=pos,
    labels=edge_labels,
    figsize=(18, 10),
    show_node_labels=True,
    # save_path=str(FIGURE_DIR / 'ground_truth_edge_labels.png'),
)


# Catalogo testuale, utile come riferimento aggiuntivo.
for edge_id, edge in edge_catalog.items():
    print(f"{edge_id}: {edge}")


In [ ]:
# ============================================================
# NET3 — GRAFO RICOSTRUITO
# Seed rappresentativo: 103, selezionato GT-free
#
# 84 TP su 119 archi reali
# 31 FP
# 35 FN
# 115 archi ricostruiti complessivi
# ============================================================

# Ground-truth edges recovered at epoch 50 (true positives).
learned_edge_ids = [
    "e00", "e01", "e02", "e03", "e04", "e05", "e06", "e07", "e08",
    "e09", "e10", "e11", "e12", "e13", "e14", "e15", "e17", "e18", "e19", "e22",
    "e23", "e24", "e25", "e26", "e27", "e28", "e29", "e31",
    "e32", "e33", "e34", "e35", "e36", "e37", "e38", "e39",
    "e40", "e41", "e42", "e44", "e45", "e46", "e48", "e49",
    "e50", "e51", "e52", "e53", "e54", "e55", "e56", "e57","e58", "e59",
    "e60", "e61",
]


# ============================================================
# FALSE POSITIVE
# 31 archi ricostruiti non appartenenti alla ground truth
# ============================================================

false_positive_edges = [
    (2, 8),
    #   (4, 10),
        # (5, 8),
          (11, 21),
            (14, 25),
    # (14, 27),
      (19, 22),
        (20, 21)
        , (24, 26),
        #   (29, 36),

    (30, 35),
    #   (32, 44),
        (39, 40),
          (40, 47)
          , (41, 43),
]
# ============================================================
# VALIDAZIONE DELLA SELEZIONE
# ============================================================

unknown_ids = sorted({
    edge_id
    for edge_id in learned_edge_ids
    if edge_id not in edge_catalog
})

if unknown_ids:
    raise ValueError(
        f"Identificativi di arco non validi: {unknown_ids}"
    )

duplicate_ids = sorted({
    edge_id
    for edge_id in learned_edge_ids
    if learned_edge_ids.count(edge_id) > 1
})

if duplicate_ids:
    raise ValueError(
        f"Identificativi ripetuti in learned_edge_ids: {duplicate_ids}"
    )

gt_set = {
    canonical_edge(edge)
    for edge in gt_edges
}

true_positive_set = {
    edge_catalog[edge_id]
    for edge_id in learned_edge_ids
}

false_positive_set = {
    canonical_edge(edge)
    for edge in false_positive_edges
}

invalid_nodes = sorted({
    node
    for edge in false_positive_set
    for node in edge
    if node not in pos
})

if invalid_nodes:
    raise ValueError(
        f"Nodi inesistenti nei falsi positivi: {invalid_nodes}"
    )

self_loops = sorted({
    edge
    for edge in false_positive_set
    if edge[0] == edge[1]
})

if self_loops:
    raise ValueError(
        f"I falsi positivi contengono self-loop: {self_loops}"
    )

false_positives_that_are_true = sorted(
    false_positive_set & gt_set
)

if false_positives_that_are_true:
    raise ValueError(
        "Questi archi appartengono alla ground truth e quindi non "
        f"sono falsi positivi: {false_positives_that_are_true}"
    )

learned_set = true_positive_set | false_positive_set

learned_edges = sorted(
    learned_set,
    key=lambda edge: (str(edge[0]), str(edge[1])),
)

false_negative_set = gt_set - learned_set

false_negative_edges = sorted(
    false_negative_set,
    key=lambda edge: (str(edge[0]), str(edge[1])),
)

false_negative_ids = [
    edge_id
    for edge_id, edge in edge_catalog.items()
    if edge in false_negative_set
]

G_learned = nx.Graph()
G_learned.add_nodes_from(pos.keys())
G_learned.add_edges_from(learned_edges)


# ============================================================
# CONTROLLI SUI CONTEGGI ATTESI
# ============================================================

# assert len(true_positive_set) == 84
# assert len(false_positive_set) == 31
# assert len(false_negative_set) == 35
# assert G_learned.number_of_edges() == 115


# ============================================================
# RIEPILOGO
# ============================================================

print(f"True positive:              {len(true_positive_set)}")
print(f"False positive:             {len(false_positive_set)}")
print(f"False negative:             {len(false_negative_set)}")
print(f"Archi ricostruiti totali:   {G_learned.number_of_edges()}")
print(f"Grafo ricostruito connesso: {nx.is_connected(G_learned)}")

print(f"ID falsi negativi: {false_negative_ids}")
print(f"Archi falsi negativi: {false_negative_edges}")
print(f"Archi falsi positivi: {sorted(false_positive_set)}")

In [ ]:
def get_common_limits(positions, margin_fraction=0.06):
    """Calcola limiti identici per entrambi i pannelli."""

    x_values = [xy[0] for xy in positions.values()]
    y_values = [xy[1] for xy in positions.values()]

    x_min = min(x_values)
    x_max = max(x_values)
    y_min = min(y_values)
    y_max = max(y_values)

    x_range = max(x_max - x_min, 1e-12)
    y_range = max(y_max - y_min, 1e-12)

    x_margin = margin_fraction * x_range
    y_margin = margin_fraction * y_range

    return (
        (x_min - x_margin, x_max + x_margin),
        (y_min - y_margin, y_max + y_margin),
    )


def draw_single_graph(
    ax,
    graph,
    positions,
    x_limits,
    y_limits,
    node_size=22,
    edge_width=0.75,
):
    """Disegna un grafo usando coordinate, limiti e scala prefissati."""

    nx.draw_networkx_edges(
        graph,
        positions,
        ax=ax,
        width=edge_width,
        edge_color="0.30",
    )

    nx.draw_networkx_nodes(
        graph,
        positions,
        ax=ax,
        node_size=node_size,
    )

    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_aspect("equal")
    ax.axis("off")


def plot_gt_vs_learned(
    ground_truth_graph,
    learned_graph,
    positions,
    figsize=(18, 8),
    node_size=22,
    edge_width=0.75,
    show_titles=False,
    save_path=None,
    dpi=300,
):
    """
    Disegna ground truth e grafo ricostruito affiancati usando:
    - le stesse coordinate;
    - gli stessi limiti;
    - la stessa scala;
    - le stesse dimensioni di nodi e archi.
    """

    x_limits, y_limits = get_common_limits(positions)

    fig, axes = plt.subplots(
        1,
        2,
        figsize=figsize,
        constrained_layout=True,
    )

    draw_single_graph(
        ax=axes[0],
        graph=ground_truth_graph,
        positions=positions,
        x_limits=x_limits,
        y_limits=y_limits,
        node_size=node_size,
        edge_width=edge_width,
    )

    draw_single_graph(
        ax=axes[1],
        graph=learned_graph,
        positions=positions,
        x_limits=x_limits,
        y_limits=y_limits,
        node_size=node_size,
        edge_width=edge_width,
    )

    if show_titles:
        axes[0].set_title(
            "Ground-truth graph",
            fontsize=14,
            pad=12,
        )
        axes[1].set_title(
            "Learned graph",
            fontsize=14,
            pad=12,
        )

    # DECOMMENTARE SOLO QUANDO SI VUOLE SALVARE LA FIGURA
    # if save_path is not None:
    #     save_path = Path(save_path)
    #     save_path.parent.mkdir(parents=True, exist_ok=True)
    #     fig.savefig(
    #         save_path,
    #         dpi=dpi,
    #         bbox_inches="tight",
    #         facecolor="white",
    #     )

    plt.show()


plot_gt_vs_learned(
    ground_truth_graph=G_gt,
    learned_graph=G_learned,
    positions=pos,
    figsize=(18, 8),
    node_size=22,
    edge_width=0.75,
    show_titles=False,
    # save_path=str(FIGURE_DIR / 'gt_vs_learned.png'),
)


In [ ]:
def compute_edge_metrics(
    ground_truth_edges,
    predicted_edges,
    nodes,
):
    """
    Calcola TP, FP, FN, TN, precision, recall, F1 e accuracy
    considerando tutte le possibili coppie di nodi di un grafo
    semplice non orientato.
    """

    gt_set_local = {
        canonical_edge(edge)
        for edge in ground_truth_edges
    }

    pred_set_local = {
        canonical_edge(edge)
        for edge in predicted_edges
    }

    node_list = list(nodes)

    all_possible_edges = {
        canonical_edge((node_list[i], node_list[j]))
        for i in range(len(node_list))
        for j in range(i + 1, len(node_list))
    }

    invalid_predicted_edges = pred_set_local - all_possible_edges
    if invalid_predicted_edges:
        raise ValueError(
            "Il grafo predetto contiene archi non validi: "
            f"{sorted(invalid_predicted_edges)}"
        )

    tp = len(gt_set_local & pred_set_local)
    fp = len(pred_set_local - gt_set_local)
    fn = len(gt_set_local - pred_set_local)
    tn = len(all_possible_edges - gt_set_local - pred_set_local)

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0

    f1 = (
        2.0 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    accuracy = (
        (tp + tn) / len(all_possible_edges)
        if all_possible_edges
        else 0.0
    )

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Accuracy": accuracy,
    }


metrics = compute_edge_metrics(
    ground_truth_edges=gt_edges,
    predicted_edges=learned_edges,
    nodes=pos.keys(),
)

for metric_name, value in metrics.items():
    if isinstance(value, float):
        print(f"{metric_name:10s}: {value:.4f}")
    else:
        print(f"{metric_name:10s}: {value}")


In [ ]:
# ============================================================
# FIGURA FINALE STILE "VERSIONE 1" SENZA TITOLI
# Ground truth a sinistra, learned graph a destra
# TP = blu, FN = arancio tratteggiato, FP = rosso
# ============================================================

import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.lines import Line2D

# ------------------------------------------------------------
# 1. Costruzione grafi
# ------------------------------------------------------------
G_gt_plot = nx.Graph()
G_gt_plot.add_nodes_from(pos.keys())
G_gt_plot.add_edges_from(gt_edges)

G_learned_plot = nx.Graph()
G_learned_plot.add_nodes_from(pos.keys())
G_learned_plot.add_edges_from(learned_edges)

# ------------------------------------------------------------
# 2. Insiemi di archi
# ------------------------------------------------------------
gt_set = {canonical_edge(e) for e in gt_edges}
learned_set = {canonical_edge(e) for e in learned_edges}

tp_edges = sorted(gt_set & learned_set)   # true positives
fn_edges = sorted(gt_set - learned_set)   # false negatives
fp_edges = sorted(learned_set - gt_set)   # false positives

# ------------------------------------------------------------
# 3. Stile grafico
# ------------------------------------------------------------
plt.rcParams["font.family"] = "DejaVu Sans"

COLOR_NODE = "#0B4F9C"
COLOR_TP   = "#2F7ED8"
COLOR_FN   = "#F28E2B"
COLOR_FP   = "#D62728"

NODE_SIZE      = 62
EDGE_WIDTH_GT  = 2.2
EDGE_WIDTH_TP  = 2.2
EDGE_WIDTH_FN  = 1.8
EDGE_WIDTH_FP  = 1.8

FIGSIZE = (18, 8.5)

# ------------------------------------------------------------
# 4. Limiti comuni per avere stessa scala e stessa inquadratura
# ------------------------------------------------------------
x_vals = [p[0] for p in pos.values()]
y_vals = [p[1] for p in pos.values()]

x_min, x_max = min(x_vals), max(x_vals)
y_min, y_max = min(y_vals), max(y_vals)

x_range = x_max - x_min
y_range = y_max - y_min

x_margin = 0.06 * x_range
y_margin = 0.06 * y_range

x_limits = (x_min - x_margin, x_max + x_margin)
y_limits = (y_min - y_margin, y_max + y_margin)

# ------------------------------------------------------------
# 5. Figura
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, constrained_layout=True)

# ---------------- LEFT: Ground truth ----------------
ax = axes[0]

nx.draw_networkx_edges(
    G_gt_plot,
    pos,
    ax=ax,
    edgelist=sorted(gt_set),
    width=EDGE_WIDTH_GT,
    edge_color=COLOR_TP,
    alpha=0.95,
)

nx.draw_networkx_nodes(
    G_gt_plot,
    pos,
    ax=ax,
    node_size=NODE_SIZE,
    node_color=COLOR_NODE,
    linewidths=0.5,
    edgecolors="white",
)

ax.set_xlim(*x_limits)
ax.set_ylim(*y_limits)
ax.set_aspect("equal")
ax.axis("off")

# ---------------- RIGHT: Learned graph ----------------
ax = axes[1]

# True positives
if len(tp_edges) > 0:
    nx.draw_networkx_edges(
        G_learned_plot,
        pos,
        ax=ax,
        edgelist=tp_edges,
        width=EDGE_WIDTH_TP,
        edge_color=COLOR_TP,
        alpha=0.95,
    )

# False negatives: archi veri mancanti, mostrati tratteggiati
if len(fn_edges) > 0:
    nx.draw_networkx_edges(
        G_gt_plot,
        pos,
        ax=ax,
        edgelist=fn_edges,
        width=EDGE_WIDTH_FN,
        edge_color=COLOR_FN,
        style="dashed",
        alpha=0.95,
    )

# False positives: archi extra, mostrati in rosso
if len(fp_edges) > 0:
    nx.draw_networkx_edges(
        G_learned_plot,
        pos,
        ax=ax,
        edgelist=fp_edges,
        width=EDGE_WIDTH_FP,
        edge_color=COLOR_FP,
        alpha=0.95,
    )

nx.draw_networkx_nodes(
    G_learned_plot,
    pos,
    ax=ax,
    node_size=NODE_SIZE,
    node_color=COLOR_NODE,
    linewidths=0.5,
    edgecolors="white",
)

ax.set_xlim(*x_limits)
ax.set_ylim(*y_limits)
ax.set_aspect("equal")
ax.axis("off")

# ------------------------------------------------------------
# 6. Legenda compatta in basso
# ------------------------------------------------------------
legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        markerfacecolor=COLOR_NODE,
        markeredgecolor="white",
        markeredgewidth=0.7,
        markersize=8,
        label="Junction"
    ),
    Line2D(
        [0], [0],
        color=COLOR_TP,
        lw=2.0,
        label="Correctly reconstructed edge"
    ),
    Line2D(
        [0], [0],
        color=COLOR_FN,
        lw=2.0,
        linestyle=(0, (4, 3)),
        label="Missing true edge"
    ),
    Line2D(
        [0], [0],
        color=COLOR_FP,
        lw=2.0,
        label="Extra (false positive) edge"
    ),
]

# fig.legend(
#     handles=legend_handles,
#     loc="lower center",
#     ncol=4,
#     frameon=False,
#     bbox_to_anchor=(0.5, -0.01),
#     fontsize=20,
#     handlelength=3.0,
#     columnspacing=2.2,
# )

# fig.savefig(
#     str(FIGURE_DIR / 'real_water.pdf'),
#     format="pdf",
#     dpi=600,
#     bbox_inches="tight",
#     facecolor="white"
# )

plt.show()

In [ ]:
# ============================================================
# FIGURA FINALE PER RETE ELETTRICA
# Ground truth a sinistra, learned graph a destra
#
# TP = conduttori ambra
# FN = blu tratteggiato
# FP = rosso
# Nodi = bus elettrici quadrati
# ============================================================

import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.lines import Line2D


# ------------------------------------------------------------
# 1. Costruzione dei grafi
# ------------------------------------------------------------

G_gt_plot = nx.Graph()
G_gt_plot.add_nodes_from(pos.keys())
G_gt_plot.add_edges_from(gt_edges)

G_learned_plot = nx.Graph()
G_learned_plot.add_nodes_from(pos.keys())
G_learned_plot.add_edges_from(learned_edges)


# ------------------------------------------------------------
# 2. Classificazione degli archi
# ------------------------------------------------------------

gt_set = {
    canonical_edge(edge)
    for edge in gt_edges
}

learned_set = {
    canonical_edge(edge)
    for edge in learned_edges
}

tp_edges = sorted(gt_set & learned_set)
fn_edges = sorted(gt_set - learned_set)
fp_edges = sorted(learned_set - gt_set)


# ------------------------------------------------------------
# 3. Stile grafico per rete elettrica
# ------------------------------------------------------------

plt.rcParams["font.family"] = "DejaVu Sans"

# Nodi e conduttori
COLOR_NODE = "#263238"          # bus elettrici
COLOR_NODE_BORDER = "#FFFFFF"

COLOR_CONDUCTOR_BORDER = "#5C4A12"
COLOR_TP = "#E0A100"            # linea correttamente ricostruita

# Errori
COLOR_FN = "#2878C8"            # linea reale mancante
COLOR_FP = "#C1121F"            # linea aggiunta erroneamente

NODE_SIZE = 58

# Gli archi corretti sono disegnati due volte:
# una linea esterna scura e una linea interna ambra.
EDGE_WIDTH_BORDER = 3.0
EDGE_WIDTH_INNER = 1.8

EDGE_WIDTH_FN = 2.1
EDGE_WIDTH_FP = 2.1

FIGSIZE = (18, 8.5)


# ------------------------------------------------------------
# 4. Limiti comuni
# ------------------------------------------------------------

x_vals = [coordinate[0] for coordinate in pos.values()]
y_vals = [coordinate[1] for coordinate in pos.values()]

x_min, x_max = min(x_vals), max(x_vals)
y_min, y_max = min(y_vals), max(y_vals)

x_range = max(x_max - x_min, 1e-12)
y_range = max(y_max - y_min, 1e-12)

x_margin = 0.06 * x_range
y_margin = 0.06 * y_range

x_limits = (
    x_min - x_margin,
    x_max + x_margin,
)

y_limits = (
    y_min - y_margin,
    y_max + y_margin,
)


# ------------------------------------------------------------
# 5. Funzioni ausiliarie
# ------------------------------------------------------------

def draw_electrical_conductors(
    graph,
    positions,
    edges,
    axis,
):
    """
    Disegna i conduttori con bordo scuro e linea interna ambra.
    """

    if not edges:
        return

    # Bordo esterno
    nx.draw_networkx_edges(
        graph,
        positions,
        ax=axis,
        edgelist=edges,
        width=EDGE_WIDTH_BORDER,
        edge_color=COLOR_CONDUCTOR_BORDER,
        alpha=0.95,
    )

    # Conduttore interno
    nx.draw_networkx_edges(
        graph,
        positions,
        ax=axis,
        edgelist=edges,
        width=EDGE_WIDTH_INNER,
        edge_color=COLOR_TP,
        alpha=1.0,
    )


def draw_electrical_nodes(
    graph,
    positions,
    axis,
):
    """
    Disegna i nodi come piccoli bus elettrici quadrati.
    """

    nx.draw_networkx_nodes(
        graph,
        positions,
        ax=axis,
        node_size=NODE_SIZE,
        node_color=COLOR_NODE,
        node_shape="s",
        linewidths=0.7,
        edgecolors=COLOR_NODE_BORDER,
    )


def configure_axis(axis):
    """
    Applica la stessa scala e inquadratura ai due pannelli.
    """

    axis.set_xlim(*x_limits)
    axis.set_ylim(*y_limits)
    axis.set_aspect("equal")
    axis.axis("off")


# ------------------------------------------------------------
# 6. Creazione della figura
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=FIGSIZE,
    constrained_layout=True,
)


# ============================================================
# PANNELLO SINISTRO: GROUND TRUTH
# ============================================================

ax = axes[0]

draw_electrical_conductors(
    graph=G_gt_plot,
    positions=pos,
    edges=sorted(gt_set),
    axis=ax,
)

draw_electrical_nodes(
    graph=G_gt_plot,
    positions=pos,
    axis=ax,
)

configure_axis(ax)


# ============================================================
# PANNELLO DESTRO: LEARNED GRAPH
# ============================================================

ax = axes[1]


# True positives
draw_electrical_conductors(
    graph=G_learned_plot,
    positions=pos,
    edges=tp_edges,
    axis=ax,
)


# False negatives
if fn_edges:
    nx.draw_networkx_edges(
        G_gt_plot,
        pos,
        ax=ax,
        edgelist=fn_edges,
        width=EDGE_WIDTH_FN,
        edge_color=COLOR_FN,
        style=(0, (5, 3)),
        alpha=0.95,
    )


# False positives
if fp_edges:
    nx.draw_networkx_edges(
        G_learned_plot,
        pos,
        ax=ax,
        edgelist=fp_edges,
        width=EDGE_WIDTH_FP,
        edge_color=COLOR_FP,
        style="solid",
        alpha=0.95,
    )


draw_electrical_nodes(
    graph=G_learned_plot,
    positions=pos,
    axis=ax,
)

configure_axis(ax)


# ------------------------------------------------------------
# 7. Legenda facoltativa
# ------------------------------------------------------------

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="s",
        color="none",
        markerfacecolor=COLOR_NODE,
        markeredgecolor=COLOR_NODE_BORDER,
        markeredgewidth=0.7,
        markersize=8,
        label="Bus",
    ),
    Line2D(
        [0],
        [0],
        color=COLOR_TP,
        markeredgecolor=COLOR_CONDUCTOR_BORDER,
        lw=2.5,
        label="Correctly reconstructed line",
    ),
    Line2D(
        [0],
        [0],
        color=COLOR_FN,
        lw=2.1,
        linestyle=(0, (5, 3)),
        label="Missing true line",
    ),
    Line2D(
        [0],
        [0],
        color=COLOR_FP,
        lw=2.1,
        label="Extra false-positive line",
    ),
]


# Decommentare per visualizzare la legenda
#
# fig.legend(
#     handles=legend_handles,
#     loc="lower center",
#     ncol=4,
#     frameon=False,
#     bbox_to_anchor=(0.5, -0.01),
#     fontsize=20,
#     handlelength=3.0,
#     columnspacing=2.2,
# )


# ------------------------------------------------------------
# 8. Salvataggio facoltativo in PDF vettoriale
# ------------------------------------------------------------

# Decommentare quando necessario
#
# from pathlib import Path
#
# Path("figures").mkdir(
#     parents=True,
#     exist_ok=True,
# )
#
fig.savefig(
    str(FIGURE_DIR / 'synth_data.pdf'),
    format="pdf",
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
)


plt.show()